Dummy Classifier (Baseline)

This notebook implements a baseline model that makes predictions using simple strategies (like always picking the most frequent class). This serves as a point of comparison for all other models.

In [1]:
import os
import pickle

import mlflow
import mlflow.sklearn
import pandas as pd
from dotenv import load_dotenv
from metrics_utils import calculate_business_metrics
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

load_dotenv()
mlflow.set_tracking_uri("https://dagshub.com/sarah-kamall/walmart-sales-classification.mlflow")
mlflow.set_experiment("Walmart-Sales-Classification")

<Experiment: artifact_location='mlflow-artifacts:/3759b8b0738c4292bfb97b1968a4dd5b', creation_time=1777646899943, experiment_id='0', last_update_time=1777646899943, lifecycle_stage='active', name='Walmart-Sales-Classification', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [2]:
train_df = pd.read_csv("../../data/model_ready/train.csv")
test_df = pd.read_csv("../../data/model_ready/test.csv")

features_selected = [
    "Size",
    "Store",
    "Dept",
    "CPI",
    "DeptFrequency",
    "Week_cos",
    "IsPreHoliday",
    "Week_sin",
    "Fuel_Price",
    "ConsumerConfRatio",
    "AvgMarkDownAmount",
]
target = "Sales_Class"
holiday_col = "IsHoliday"

X_train = train_df[features_selected]
y_train = train_df[target]
X_test = test_df[features_selected]
y_test = test_df[target]

# We need IsHoliday for business metrics evaluation
test_holidays = test_df[holiday_col]

In [3]:
with mlflow.start_run(run_name="Dummy_Baseline"):
    model_path = "dummy_model.pkl"
    if os.path.exists(model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        print("Model loaded from pickle")
    else:
        # 1. Initialize and Train
        model = DummyClassifier(strategy="stratified", random_state=42)
        model.fit(X_train, y_train)
        with open(model_path, "wb") as f:
            pickle.dump(model, f)
        print("Model trained and saved to pickle")

Model loaded from pickle


🏃 View run Dummy_Baseline at: https://dagshub.com/sarah-kamall/walmart-sales-classification.mlflow/#/experiments/0/runs/f39b6b7f8950462ea0a8cf0bdfc8a557
🧪 View experiment at: https://dagshub.com/sarah-kamall/walmart-sales-classification.mlflow/#/experiments/0


In [4]:
y_pred = model.predict(X_test)

In [5]:
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

In [6]:
# 4. Calculate Business Metrics
biz_metrics = calculate_business_metrics(y_test, y_pred, test_holidays)

In [7]:
mlflow.log_param("model_type", "DummyClassifier")
mlflow.log_param("strategy", "stratified")

mlflow.log_metric("accuracy", acc)
mlflow.log_metric("f1_weighted", f1)
mlflow.log_metric("holiday_accuracy", biz_metrics["holiday_accuracy"])
mlflow.log_metric("weighted_classification_error", biz_metrics["weighted_classification_error"])

In [8]:
# 6. Save Model Artifact
mlflow.log_artifact(model_path)

mlflow.sklearn.log_model(model, artifact_path="model")

print(f"Baseline Accuracy: {acc:.4f}")
print(f"Holiday Accuracy: {biz_metrics['holiday_accuracy']:.4f}")
print(f"Weighted Error: {biz_metrics['weighted_classification_error']:.4f}")

2026/05/03 23:42:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Baseline Accuracy: 0.4990
Holiday Accuracy: 0.5009
Weighted Error: 0.4994
